# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [ ]:
# try:
#     !pip install gurobipy
# except:
#     %pip install gurobipy

In [ ]:
# test

In [ ]:
#!git clone -b feature/shift-objects https://github.com/poprjaduhhaa/Modellierungsseminar-Firestation.git
#%cd /content/Modellierungsseminar-Firestation/coding
#We dont need it anmore since Colab is not uswed anzmore



fatal: destination path 'Modellierungsseminar-Firestation' already exists and is not an empty directory.


[WinError 3] Das System kann den angegebenen Pfad nicht finden: '/content/Modellierungsseminar-Firestation/coding'
c:\Users\dirkb\Documents\GitHub\Modellierungsseminar-Firestation\coding


In [ ]:
# install Gurobi package in case not done yet:
%pip install gurobipy 
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import datetime as dt
from pathlib import Path # for easier and robust folder and file handling across OS (Path can be used by Pandas directly)
import src.Shift as Shift # tailor-made data type for shift definitions

Note: you may need to restart the kernel to use updated packages.


### inputs and parameters

In [ ]:
# global static variables

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = 365  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
NB_CYLCEs = 1 # number of cycles (for split by qualification)

DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7
                 }
DICT_WEEKDAYS_RETURN = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday", 5: "Friday", 6: "Saturday", 7: "Sunday"}

# determine folder structure for inputs and outputs
PROJECT_ROOT = Path.cwd() # main folder of the code
FOLDER_INPUT =  PROJECT_ROOT / "input" # data input
FOLDER_LOGS = PROJECT_ROOT / "logs" # folder for log files, eg exorts of data sets for more transparency
FOLDER_OUTPUT = PROJECT_ROOT / "output/"
FOLDER_AND_FILE_LOG =  PROJECT_ROOT / "logs" / "cyclePlanning_logs.txt"

### function writeToLogs

In [ ]:
# LOGS / for process transparency 
def writeToLogs(yourStatusMessage:str, file, deleteHistory=False):
    try:
        if deleteHistory:
            with file.open("w") as log:
                log.write(dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
                log.write(" // ")
                log.write(yourStatusMessage+"\n")
        else:
            with file.open("a") as log:
                log.write(dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
                log.write(" // ")
                log.write(yourStatusMessage+"\n")
    except FileNotFoundError:
        print(f"file '{file}' not found - I skip logging and go on with my work...")
    except PermissionError:
        print(f"file '{file}' is locked for editing - I skip logging and go on with my work...")



### Read parameters

In [6]:
# determine folder structure for inputs and outputs
PROJECT_ROOT = Path.cwd() # main folder of the code
FOLDER_INPUT =  PROJECT_ROOT / "input" # data input
FOLDER_LOGS = PROJECT_ROOT / "logs" # folder for log files, eg exorts of data sets for more transparency
FOLDER_OUTPUT = PROJECT_ROOT / "output/"
FOLDER_AND_FILE_LOG =  PROJECT_ROOT / "logs" / "cyclePlanning_logs.txt"
writeToLogs("STARTED the cycle planning process", FOLDER_AND_FILE_LOG, True)

In [7]:
def readParameters(filename, mySep=";") -> dict:
    writeToLogs(f"reading parameters from file {filename}", FOLDER_AND_FILE_LOG)
    df = pd.read_csv(filename, sep=mySep, dtype=str)
    params = dict(zip(df["parameter"], df["value"]))
    writeToLogs("parameters loaded", FOLDER_AND_FILE_LOG)
    return params

# load parameters from CSV into dict
params = readParameters(FOLDER_INPUT / "parameters.csv")

### Global variables

In [8]:
# global static variables

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(params["max_cycle_length"])   # max number of cycle weeks
MIN_REST        = int(params["min_rest"])            # min rest time between shifts in minutes
MAX_CONSEC_DAYS = int(params["max_conseq_working_days"])  # max consecutive working days

# variables for hourly fairness and average weekly work hours
AVG_WEEKLY_HOURS    = float(params["avg_weekly_work_hours"])   # target avg weekly work hours
AVG_REFERENCE_WEEKS = int(params["avg_reference_weeks"])       # weeks window for average
MAX_WEEKLY_HOURS    = AVG_WEEKLY_HOURS * AVG_REFERENCE_WEEKS   # total hours over reference window

#MAX_CYCLE_WEEKS = 365  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
#NB_CYLCEs = 1 # number of cycles (for split by qualification)

DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7
                 }
DICT_WEEKDAYS_RETURN = {1: "Monday", 2: "Tuesday", 3: "Wednesday", 4: "Thursday", 5: "Friday", 6: "Saturday", 7: "Sunday"}



### function writeDataToLogs

In [ ]:
# LOG / for transparency write shift objects into a file:
def writeDataToLogs(data, filename):
    try:
        pd.DataFrame(data).to_csv(filename, sep=";", index=True, encoding="utf-8", decimal=".")
    except PermissionError:
        print("log file for shift_object is open - I skipped saving and executed succeeding code")

### function readShiftSet

In [ ]:
# read input data
# shift set

def readShiftSet(filename, mySep: str=";") -> pd.DataFrame:
    writeToLogs(f"reading shift data set from file {filename}",FOLDER_AND_FILE_LOG)
    input_data = pd.read_csv(filename, sep=mySep, dtype=str) # import all values as string as first step
    # adjust data types for columns not (supposed to be) reflecting strings
    input_data["shift_required_staff"].astype(int)
    input_data["shift_class"].astype(int)
    input_data["shift_work_time_assignment"].astype(float)
    input_data["isWorkShift"] = input_data["isWorkShift"].astype(int).astype(bool)
    # multiply rows by the number of required workers (.explode())
    #input_data = input_data.assign(shift_required_staff=input_data["shift_required_staff"].apply(lambda n: list(range(1, n+1)))).explode("shift_required_staff")

    writeToLogs("import successfull",FOLDER_AND_FILE_LOG)

    #multiply rows by requirement
    input_data = input_data[input_data["shift_required_staff"].notna() 
                            & (input_data["shift_required_staff"] != 0)].loc[lambda x: x.index.repeat(x["shift_required_staff"])].assign(shift_ID=lambda x: x.groupby(level=0).cumcount().add(1)
                                   .astype(str)
                                   .radd("_")
                                   .radd(x["shift_ID"]))    
    
    return input_data
    # potentially add further data cleaning steps


### function build_shift_objects

In [ ]:

# added objects to read whole csv file
def build_shift_objects(df: pd.DataFrame) -> list:
    shift_objects = []
    for _, row in df.iterrows():
        s = Shift.Shift(
            shift_id=row['shift_ID'],
            description=row['shift_details'],
            weekdays=[d.strip() for d in row['shift_weekdays'].split(',')],
            start=dt.time(*map(int, row['shift_start_time'].split(':'))),
            end=dt.time(*map(int, row['shift_end_time'].replace('24','0').split(':'))),
            required_staff=0 if row['shift_required_staff'] == 'none' else int(row['shift_required_staff']),
            shift_class=int(row['shift_class']),
            shift_work_time_assignment=str(row['shift_work_time_assignment']),
            is_work_shift=bool(row['isWorkShift']),
            required_qualification=row['[shift_required_qualification]']
        )
        shift_objects.append(s)

    writeToLogs("list of shift_objects built",FOLDER_AND_FILE_LOG)
    return shift_objects


In [ ]:

# basic inputs and parameters

Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday
#Shifts = ["frueh", "spaet", "nacht", "frei"] # including free shifts
#WorkShifts = ["frueh", "spaet", "nacht"] # excluding free shifts

# improvements outstanding:
    # use files for parameter input
        # shift definitions
        # available staff
        # user objectives: weighted priorities

creating shift objects:

In [ ]:

# read input data for shift definitions
data_shiftSet = readShiftSet(FOLDER_INPUT / "input_ShiftDataSet_Pesch.csv")
#data_shiftSet = readShiftSet(FOLDER_INPUT / "input_ShiftDataSet.csv")

shift_objects = build_shift_objects(data_shiftSet)
print(type(shift_objects))
#shift_objects = shift_objects.loc[shift_objects.index.repeat(shift_objects["required_staff"])] # multiply by required staff


WorkShifts = [s.shift_id for s in shift_objects] #object oriented solution
writeToLogs(f"WorkShifts are defined as {WorkShifts}", FOLDER_AND_FILE_LOG)

freeDayShift = Shift.Shift("[freeDay]", 
                           "dummy shift for free days", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           "06:00", 
                           "06:00", 
                           0, 
                           1, 
                           0, 
                           False, 
                           None 
                           )
shift_objects.append(freeDayShift)

spareShift = Shift.Shift("[spareShift]", 
                           "dummy for spare shifts", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           "06:00", 
                           "06:00", 
                           1, 
                           5, 
                           0, 
                           True, 
                           None 
                           )
shift_objects.append(spareShift)

#log
writeDataToLogs(shift_objects, FOLDER_LOGS / "log_shift_object.csv")

Shifts = [s.shift_id for s in shift_objects]
writeToLogs(f"    Shifts are defined as {Shifts}", FOLDER_AND_FILE_LOG)

print(Shifts)


<class 'list'>
['[00day000week]_1', '[00day000week]_2', '[00day000week]_3', '[00day000week]_4', '[00day000week]_5', '[00day000week]_6', '[00day000week]_7', '[night000week]_1', '[night000week]_2', '[night000week]_3', '[night000week]_4', '[night000week]_5', '[00dayweekend]_1', '[00dayweekend]_2', '[00dayweekend]_3', '[00dayweekend]_4', '[00dayweekend]_5', '[nightweekend]_1', '[nightweekend]_2', '[nightweekend]_3', '[nightweekend]_4', '[nightweekend]_5', '[freeDay]', '[spareShift]']


In [14]:
# Pre-compute all shift pairs that violate MIN_REST if scheduled on consecutive days.
# Excludes dummy shifts (freeDay, spareShift) since they have no real start/end times.
# Result: list of (sh1_id, sh2_id) tuples that cannot appear on consecutive days in a snake.
DUMMY_SHIFTS = {"[freeDay]", "[spareShift]"}
incompatible_pairs = [
    (sh1.shift_id, sh2.shift_id)
    for sh1 in shift_objects if sh1.shift_id not in DUMMY_SHIFTS
    for sh2 in shift_objects if sh2.shift_id not in DUMMY_SHIFTS
    if Shift.rest_minutes_between(sh1, sh2) < MIN_REST
]

In [15]:
# Pre-compute work hours per shift using shift_duration_hours from Shift.py.
# Used in the weekly work time constraint.
# freeDay and spareShift are excluded — they have no real duration.
shift_hours = {
    s.shift_id: Shift.shift_duration_hours(s)
    for s in shift_objects
    if s.shift_id not in DUMMY_SHIFTS
}

### modelling

In [16]:
# modelling

m = gp.Model("SnakeBuilding_simple")

# variables:
# x[s, d, sh] = 1, when snake s is working in shift sh on day d
x = m.addVars(MAX_CYCLE_WEEKS, Weekdays, Shifts, vtype=GRB.BINARY, name="x")

# active[s] = 1, when snake s is used
active = m.addVars(MAX_CYCLE_WEEKS, vtype=GRB.BINARY, name="active") # all other snakes are used as placeholders but not necessarily get activated


### conditions:

#### condition c01
_(idea is to use a unique ID for each condition for better reference)_

each shift has to be covered on each day

$$\sum_{s=1}^{n}{x_{s,d,w}} >= 1    \forall d \in D, \forall w \in W$$

$x_{s,d,w} = 1$, when snake s is working in work shift w on day d

$x: $ binary variable,
$s: $ snake number,
$d: $ weekday,
$ws: $ work shift


In [17]:
# SUBJECT TO:

for d in Weekdays:
    for ws in WorkShifts:
        shift = next(s for s in shift_objects if s.shift_id == ws)
        if DICT_WEEKDAYS[shift.weekdays[0]] <= d <= DICT_WEEKDAYS[shift.weekdays[-1]]:
            m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
                        name=f"Cover_day{d}_{ws}")

#Логика: перед добавлением ограничения проверяем входит ли день d в список weekdays этой смены. Если нет ограничение не добавляется, смена в этот день не требуется.

# 1. each shift has to be covered on each day
#for d in Weekdays:
#    for ws in WorkShifts:
#        m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
#                    name=f"Cover_day{d}_{ws}")


####

#### condition c02

In [18]:
# 2. each snake can have at most one shift per day
for s in range(MAX_CYCLE_WEEKS):
    for d in Weekdays:
        m.addConstr(gp.quicksum(x[s, d, sh] for sh in Shifts) == active[s],
                    name=f"OneShiftPerDay_s{s}_d{d}")


#### condition c03

In [ ]:

# 3) at max 5 consecutive working days (ensure time for resting)
for s in range(MAX_CYCLE_WEEKS):
    for start in range(1, 3):
        m.addConstr(
            gp.quicksum(x[s, d, sh] for d in range(start, start + MAX_CONSEC_DAYS+1) #window must be 6 days: checking 5 out of 5 always passes, we need the extra day to catch violations
                        for sh in WorkShifts) <= MAX_CONSEC_DAYS,
            name=f"Max{MAX_CONSEC_DAYS}Work_s{s}_start{start}"
        )



#### condition c04  

ensure that cycle weeks are activated in ascending order  

$$y_s - y_{s+1} >= 0$$

In [20]:
# 4) ensure that cycle weeks are activated in ascending order (not like 2-5-9-17-29-.... but 1-2-3-4-....)

for s in range(MAX_CYCLE_WEEKS-1):
    m.addConstr(active[s]>=active[s+1])

#### Condition c05

In [21]:
# c05: minimum rest time between consecutive shifts within a snake week.
# If sh1 on day d and sh2 on day d+1 violate MIN_REST, they cannot both be assigned to the same snake.
# Covers days 1-6 only; wrap-around (day 7 -> day 1) not yet modelled.
for s in range(MAX_CYCLE_WEEKS):
    for d in range(1, 7):
        for (sh1, sh2) in incompatible_pairs:
            m.addConstr(
                x[s, d, sh1] + x[s, d+1, sh2] <= 1,
                name=f"MinRest_s{s}_d{d}_{sh1}_{sh2}"
            )

#### Condition c06

In [22]:
# c06: total work hours per active snake week must not exceed MAX_WEEKLY_HOURS.
# Ensures no snake week accumulates more than the allowed weekly work time.
# Bound scales with active[s] so inactive snakes are not constrained.
for s in range(MAX_CYCLE_WEEKS):
    m.addConstr(
        gp.quicksum(
            shift_hours.get(sh, 0) * x[s, d, sh]
            for d in Weekdays
            for sh in Shifts
            if sh in shift_hours
        ) <= MAX_WEEKLY_HOURS * active[s],
        name=f"MaxWeeklyHours_s{s}"
    )

### objective

In [23]:
# set objective function: minimize number of active snakes
m.setObjective(gp.quicksum(active[s] for s in range(MAX_CYCLE_WEEKS)), GRB.MINIMIZE)

# improvements outstanding:
    # add various weighted objectives => based on user input

#run optimizer
m.optimize()



Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Pop!_OS 24.04 LTS")

CPU model: Intel(R) Core(TM) i5-7300HQ CPU @ 2.50GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 4 logical processors, using up to 4 threads

Academic license 2810721 - for non-commercial use only - registered to ar___@student.uni-siegen.de
Optimize a model with 486259 rows, 61685 columns and 1234428 nonzeros (Min)
Model fingerprint: 0x0481a1ad
Academic license 2806446 - for non-commercial use only - registered to di___@student.uni-siegen.de
Optimize a model with 3803 rows, 61685 columns and 201113 nonzeros (Min)
Model fingerprint: 0x12f374e0
Model has 365 linear objective coefficients
Variable types: 0 continuous, 61685 integer (61685 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 471945 rows and 32120 columns
Presolve time: 0.53s
Presolved: 14314

### results

In [ ]:
# output (raw version, to be improved for better readability)
    # improvements outstanding:
        # write results in file
        # create a shift overview per staff member

if m.status == GRB.OPTIMAL:
    print("\nminimum number of cycle weeks:", int(m.objVal))
    writeToLogs(f"successfully finished cycle plan: found an optimal solution using {int(m.objVal)} cycle weeks",FOLDER_AND_FILE_LOG)
    for s in range(MAX_CYCLE_WEEKS):
        if active[s].X > 0.5: # > 0.5 instead of == 1: Gurobi stores binary vars as floats, 0.9999 would fail an exact equality check
            print(f"\ncycle week {s+1}:")
            writeToLogs(f"\ncycle week {s+1}:",FOLDER_AND_FILE_LOG)
            for d in Weekdays:
                for sh in Shifts:
                    if x[s, d, sh].X > 0.5: # > 0.5 instead of == 1: Gurobi stores binary vars as floats, 0.9999 would fail an exact equality check
                        print(f"\tday {d}: {sh}")
                        writeToLogs(f"\tday {d}: {sh}",FOLDER_AND_FILE_LOG)

# output to file
if m.status == GRB.OPTIMAL:
    output_string = ""
    writeToLogs(f"writing solution to file {FOLDER_OUTPUT} 'output_cycle.csv'",FOLDER_AND_FILE_LOG)
    with open(FOLDER_OUTPUT / "output_cycle.csv", "w") as file:
        file.write("FINAL CYCLE:\nMon;Tue;Wed;Thu;Fri;Sat;Sun;\n")
    for s in range(MAX_CYCLE_WEEKS):
        if active[s].X > 0.5: # > 0.5 instead of == 1: Gurobi stores binary vars as floats, 0.9999 would fail an exact equality check
            for d in Weekdays:
                for sh in Shifts:
                    if x[s, d, sh].X > 0.5: # > 0.5 instead of == 1: Gurobi stores binary vars as floats, 0.9999 would fail an exact equality check
                        output_string = output_string + sh + ";"
            with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file: 
                file.write(output_string + "\n")
                output_string = ""



minimum number of cycle weeks: 17

cycle week 1:
	day 1: [00day000week]_5
	day 2: [00day000week]_2
	day 3: [night000week]_4
	day 4: [spareShift]
	day 5: [night000week]_5
	day 6: [spareShift]
	day 7: [nightweekend]_3

cycle week 2:
	day 1: [00day000week]_4
	day 2: [night000week]_1
	day 3: [spareShift]
	day 4: [00day000week]_6
	day 5: [night000week]_3
	day 6: [spareShift]
	day 7: [nightweekend]_1

cycle week 3:
	day 1: [00day000week]_6
	day 2: [night000week]_4
	day 3: [spareShift]
	day 4: [00day000week]_2
	day 5: [night000week]_1
	day 6: [spareShift]
	day 7: [nightweekend]_3

cycle week 4:
	day 1: [night000week]_1
	day 2: [spareShift]
	day 3: [00day000week]_3
	day 4: [00day000week]_3
	day 5: [night000week]_4
	day 6: [spareShift]
	day 7: [00dayweekend]_1

cycle week 5:
	day 1: [00day000week]_2
	day 2: [00day000week]_6
	day 3: [night000week]_3
	day 4: [spareShift]
	day 5: [night000week]_2
	day 6: [spareShift]
	day 7: [nightweekend]_5

cycle week 6:
	day 1: [00day000week]_1
	day 2: [00day0

In [25]:
print(shift_objects[1])

Shift(shift_id='[00day000week]_2', description='day shift week', weekdays=['Mon', 'Tue', 'Wed', 'Thu', 'Fri'], start=datetime.time(6, 0), end=datetime.time(16, 45), required_staff=7, shift_class=2, shift_work_time_assignment='nan', is_work_shift=True, required_qualification='none')


In [26]:
print(shift_hours)

{'[00day000week]_1': 10.75, '[00day000week]_2': 10.75, '[00day000week]_3': 10.75, '[00day000week]_4': 10.75, '[00day000week]_5': 10.75, '[00day000week]_6': 10.75, '[00day000week]_7': 10.75, '[night000week]_1': 14.75, '[night000week]_2': 14.75, '[night000week]_3': 14.75, '[night000week]_4': 14.75, '[night000week]_5': 14.75, '[00dayweekend]_1': 10.75, '[00dayweekend]_2': 10.75, '[00dayweekend]_3': 10.75, '[00dayweekend]_4': 10.75, '[00dayweekend]_5': 10.75, '[nightweekend]_1': 14.75, '[nightweekend]_2': 14.75, '[nightweekend]_3': 14.75, '[nightweekend]_4': 14.75, '[nightweekend]_5': 14.75}
